# IR/VIS residual maps: finding rhombohedral graphite

This notebook runs the tool on the two demo flakes, then on your own measurements.

**How to use it:** click into a grey code cell and press **Shift + Enter** to run it. Run the cells from top to bottom. A cell is running while it shows `[*]` on the left.

What the tool makes for each flake:
1. the **visible** image;
2. the **corrected IR** image (contrast to the substrate, with dark frame, flat field and background removed);
3. the **residual**: IR minus the IR that *hexagonal* graphite of the same visible colour would have. **Blue areas are rhombohedral candidates.**

How to take the images: `docs/IMAGING_PROTOCOL.md` and `docs/MEASUREMENT_CHECKLIST.md`.

## 1. Check the installation

This checks that the required Python packages are installed. If something is missing, run the cell after it once, then restart the kernel (menu **Kernel → Restart Kernel**) and start again from the top.

In [ ]:
import importlib.util, sys

missing = [m for m in ("numpy", "scipy", "skimage", "tifffile", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    print("Missing packages:", missing)
    print("Run the next cell once, then restart the kernel.")
else:
    print("All packages are installed (Python " + sys.version.split()[0] + ").")

In [ ]:
# Only needed if the cell above reported missing packages: remove the '#' in the next line and run this cell.
# %pip install -r requirements.txt

In [ ]:
import residuals      # the tool itself: residuals.py, in the same folder as this notebook
print("Tool loaded.")

## 2. Demo: flake 191

The demo flakes are in the `demo` folder. Each run takes about 10 seconds. The results are saved in `demo/191/residuals/`.

In [ ]:
r191 = residuals.run("demo/191")
residuals.show(r191)

**How to read the residual (right panel):**
- **Blue** = darker in IR than hexagonal graphite of the same colour = **rhombohedral candidate**. Here: the domains in the centre and on the left, confirmed by Raman.
- The colour scale is centred on the flake's median. Compare neighbouring areas of the **same visible colour** (thickness).
- Typical signal: −0.015 to −0.02 on thick, dark-blue parts; −0.004 to −0.01 on purple parts; about 0 on thin, light-purple parts.
- Not rhombohedral: thin lines along folds and wrinkles, the outer 1–2 pixels of the flake edge, isolated dark dots (dust).
- **Grey** = left out (saturated parts, or colours outside the calibration).

## 3. Demo: flake 80_02, with the thick part

`part="thick"` also makes a second set of figures for the thick part only, outlined in yellow, with the colour scale centred on that part. Use `part="thin"` for the thin part.

In [ ]:
r80 = residuals.run("demo/80_02", part="thick")
residuals.show(r80)

## 4. Your own flake

Put the images of one flake in one folder:

| File name | What |
|---|---|
| `vis.tif` | visible image (RGB) |
| `ir1.tif` | IR image (raw 16-bit) |
| `ir2.tif` | optional: the same flake shifted by about ¼ of the field |
| `dark1.tif`, `dark2.tif`, … | IR with the lamp blocked (**required**) |
| `flat1.tif` … `flat8.tif` | IR of bare substrate (strongly recommended) |

The names only need to **start** with `dark` (or `black`) and `flat`; the other names are free. The RGB image is recognised as the visible image, all other images as IR.

**Set the folder below.** On Windows: in File Explorer, click the folder, press **Ctrl + Shift + C** (or Shift + right-click → *Copy as path*), paste it between the quotes after `r`, and remove the extra quotes. Keep the `r` in front: it lets Windows paths with `\` work.

In [ ]:
FLAKE_FOLDER = r"C:\Users\yourname\Documents\flakes\my_flake"    # <- change this to your flake folder
PART = None      # None, "thick" or "thin"

import os
if os.path.isdir(FLAKE_FOLDER):
    r = residuals.run(FLAKE_FOLDER, part=PART)
    residuals.show(r)
else:
    print("Folder not found. Set FLAKE_FOLDER in this cell to your flake folder, then run the cell again.")

The results are in a new `residuals` sub-folder of the flake folder:

| File | Contents |
|---|---|
| `<name>_residual.svg` | the three panels together |
| `<name>_1_visible.svg`, `_2_ir_corrected.svg`, `_3_residual.svg` | the same panels, one per file |
| `<name>_summary.txt` | files used, residual level, **warnings** |
| `data/<name>_residual.tif` etc. | the numbers (32-bit float TIFF), for your own analysis |

The SVG files open in any web browser, Inkscape, PowerPoint or Word, and stay sharp at any size.

**Warnings** (printed under the figures and in the summary) point to settings to fix in the next measurement, e.g. VIS too bright, white balance off, no flat field, too little bare substrate.

## 5. Many flakes at once

Put every flake in its own sub-folder of one parent folder, and set the parent folder below. A flake with missing files is skipped with a message.

In [ ]:
PARENT_FOLDER = r"C:\Users\yourname\Documents\flakes"    # <- change this
PART = None

if os.path.isdir(PARENT_FOLDER):
    results = residuals.run_batch(PARENT_FOLDER, part=PART)
    residuals.show(results)
else:
    print("Folder not found. Set PARENT_FOLDER in this cell, then run the cell again.")

## 6. Files with other names (optional)

If your files are not in the folder layout above, give them one by one. `dark` and `flats` can be a list of files or a pattern with `*`.

In [ ]:
# r = residuals.run(vis=r"C:\data\flake7_VIS.tif",
#                   ir=r"C:\data\flake7_IR_A.tif",
#                   ir2=r"C:\data\flake7_IR_B.tif",          # optional
#                   dark=r"C:\data\calibration\dark*.tif",
#                   flats=r"C:\data\calibration\flat*.tif",
#                   out=r"C:\data\results\flake7", name="flake7")
# residuals.show(r)

## 7. Working with the numbers

`run()` returns the arrays too. This example uses the demo flake 191: it prints the residual statistics and draws a histogram of the residual values on the flake.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

res = r191["result"]["residual"]                 # residual on the IR pixel grid (NaN off the flake)
vals = res[np.isfinite(res)]
print(f"flake pixels: {vals.size}, median {np.median(vals):+.4f}, "
      f"share below median - 0.005: {np.mean(vals < np.median(vals) - 0.005):.1%}")

plt.figure(figsize=(6, 3.5))
plt.hist(vals, bins=200, range=(-0.04, 0.04))
plt.xlabel("residual (IR contrast)"); plt.ylabel("pixels"); plt.title("flake 191")
plt.show()

Other arrays in `r["result"]`: `ir_c` (corrected IR contrast), `rgb` (VIS contrast per channel), `flake` (flake mask), `excluded` (left-out pixels). They are in the IR camera orientation; the saved figures and TIFFs in `data/` are rotated by 180° to match the visible image.

## Options

| Option | Meaning |
|---|---|
| `part="thick"` / `"thin"` | extra figures for the thick (or thin) part only |
| `part_by="ir"` / `"vis"` | find the part by IR contrast or by visible brightness (default `"auto"`) |
| `no_thick=True` | keep pixels outside the calibrated colour range (switched on automatically when more than half the flake is outside it) |
| `ir_orientation=True` | keep the IR camera orientation (default: rotated by 180° to match the visible image) |
| `out=...`, `name=...` | output folder and flake name |
| `calib=...` | another calibration file (default `calib/calib.npz`) |

## If something goes wrong

- **`ModuleNotFoundError: No module named 'residuals'`:** the notebook must be opened from the tool's folder (the one with `residuals.py`). Start Jupyter in that folder.
- **`folder not found`:** check the path; keep the `r` in front of Windows paths.
- **`no dark frame found`:** a file starting with `dark` (or `black`) must be in the flake folder.
- **`expected exactly one VIS (RGB) image`:** remove extra RGB images (screenshots, exports) from the flake folder.
- **Files in OneDrive / SharePoint:** make sure they are downloaded (right-click → *Always keep on this device*), otherwise reading can be very slow.